# RoBERTa AI-vs-Human classifier on PolitiFact++ & GossipCop++

Fine-tunes **`roberta-base`** (the best of the three models in
`gptvshumantext-3top-models-results (1).ipynb`) on the LIFE benchmarks. **Task = provenance:**
LLM-generated (MF + MR) = **AI (1)** vs human-written (HF + HR) = **human (0)** — the same
labeling as the TF-IDF baseline, but with a transformer that reads style/syntax/context instead
of word counts, so it should recover the AI-vs-human signal that bag-of-words missed.

- **GPU runtime required:** Runtime → Change runtime type → GPU (a free **T4** is plenty —
  `roberta-base` is 125M params; minutes, not hours).
- 3 epochs, AdamW 3e-5, stratified 70/15/15 split, 3 seeds → results are **mean ± std**.
- **PolitiFact++** has only ~78 test articles per seed → noisy; read the mean ± std. **GossipCop++**
  (~3k test) is stable.
- Still provenance, **not** LIFE's fake-news task. For a LIFE-comparable cut, relabel to MF-vs-MR
  by editing `FILE_LABELS` in `run_life_baseline.py`.

In [ ]:
!pip install -q transformers  # torch / scikit-learn are preinstalled on Colab

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime to GPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PROJECT_DIR    = '/content/drive/MyDrive/LIFE'
DATASET_ROOT   = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset'
POLITIFACT_DIR = f'{DATASET_ROOT}/PolitiFact++'
GOSSIPCOP_DIR  = f'{DATASET_ROOT}/GossipCop++'

os.chdir(PROJECT_DIR)  # so the relative script path below resolves
print('PolitiFact++ found:', os.path.isdir(POLITIFACT_DIR))
print('GossipCop++  found:', os.path.isdir(GOSSIPCOP_DIR))

## Run the classifier

Each cell prints per-seed training progress + a test `classification_report` and confusion
matrix, then a **mean ± std** summary across seeds. PolitiFact++ is quick; GossipCop++
(3 seeds × 3 epochs over ~14k train articles) is the longer run — minutes on a T4/A100.

In [ ]:
!python ai_vs_human_code/run_life_roberta.py --data_dir "{POLITIFACT_DIR}" --name PolitiFact++

In [ ]:
!python ai_vs_human_code/run_life_roberta.py --data_dir "{GOSSIPCOP_DIR}" --name GossipCop++

## Notes
- **Labels are provenance, not veracity:** AI = MF/MR (GPT-3.5), human = HF/HR — same as the
  TF-IDF baseline, so the two are directly comparable. Expect a large jump over TF-IDF
  (which collapsed to ~0.65 / 0.72 accuracy with poor AI recall).
- Config mirrors the source notebook: `roberta-base`, AdamW 3e-5, batch 32, 3 epochs, stratified
  70/15/15, seeds 7/42/123. Truncation is `--max_length 512` (bumped from the notebook's 256
  since news articles are longer). No model checkpoints are saved (baseline run).
- Tweak via CLI, e.g. `--seeds 42` for a single quick run, or `--epochs 2 --max_length 256` to
  speed up GossipCop++.